# Filters, Delays and DSP

One-pole/two-pole filters, delay lines, and the `pysndlib.dsp` module.

Docs:
- [CLM generators](https://testcase.github.io/pysndlib/clm_generators.html)
- [other generators](https://testcase.github.io/pysndlib/other_generators.html)
- Original: [sndclm.html](https://ccrma.stanford.edu/software/snd/snd/sndclm.html)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

import pysndlib.clm as clm
import pysndlib.sndlib as sndlib

SR = 44100

def mag(x, n=4096):
    rl = np.copy(x[:n].astype(float)); im = np.zeros(n)
    clm.mus_fft(rl, im, n, 1)
    clm.mus_rectangular2polar(rl, im)
    return np.abs(rl[:n//2])

## One-pole: the workhorse lowpass

`make_one_pole(a0, b1)` implements y(n) = a0*x(n) - b1*y(n-1); the DC gain is a0/(1+b1). Watch white noise get darker:

In [ ]:
noise = np.random.default_rng(0).uniform(-1, 1, SR)
lp = clm.make_one_pole(0.1, -0.9)          # DC gain 1, very smooth
out = np.array([clm.one_pole(lp, float(x)) for x in noise])

f = np.arange(2048) * SR / 4096
plt.figure(figsize=(14, 3))
plt.plot(f, mag(noise), alpha=0.4, label='input')
plt.plot(f, mag(out), label='filtered')
plt.xlim(0, 8000); plt.legend(); plt.show()

## Formants: vocal-tract resonances

`formant` is a two-pole resonator. A bank of them driven by noise approximates vowel sounds -- the heart of CLM's voice instruments:

In [ ]:
def vowel(formants, f0=120.0, dur=1.0, out='vowel.wav'):
    with clm.Sound(out, channels=1, srate=SR):
        src = clm.make_rand(f0)                        # glottal-ish source
        bank = clm.make_formant_bank([clm.make_formant(fr, 0.99) for fr in formants],
                                     [1/len(formants)] * len(formants))
        for i in range(int(dur * SR)):
            clm.outa(i, clm.formant_bank(bank, clm.rand(src)))

vowel([730, 1090, 2440])     # roughly "ah"

In [ ]:
ipd.Audio('vowel.wav', rate=SR)

## Delay lines

`delay` holds past inputs -- echo, flanging and comb resonance are all delays in disguise (see [the delays notebook](11_Delays_Physical.ipynb)):

In [ ]:
with clm.Sound('echo.wav', channels=1, srate=SR):
    d = clm.make_delay(SR // 5)             # 0.2 s echo
    osc = clm.make_oscil(440.0)
    amp = clm.make_env([0, 0, 10, 1, 100, 0], duration=0.3)
    for i in range(SR):
        v = clm.env(amp) * clm.oscil(osc)
        clm.outa(i, v + 0.6 * clm.delay(d, v))   # mix in the delayed signal

In [ ]:
d, _ = clm.file2ndarray('echo.wav')
plt.figure(figsize=(14, 2.5))
plt.plot(d[0], lw=0.6); plt.title('a tone and its echo'); plt.show()

## The dsp module

`pysndlib.dsp` holds ready-made filter designs (biquads, Butterworths, FIR helpers) translated from the original dsp.scm:

In [ ]:
import pysndlib.dsp as dsp

noise = np.random.default_rng(1).uniform(-1, 1, SR)
hp = dsp.make_butter_high_pass(2000.0)
out = np.array([dsp.butter(hp, float(x)) for x in noise])

f = np.arange(2048) * SR / 4096
plt.figure(figsize=(14, 3))
plt.plot(f, mag(out), label='highpassed noise')
plt.xlim(0, 8000); plt.legend(); plt.show()

## Processing files

The package ships example sounds in `pysndlib.examples`; `down_oct` transposes one down by n octaves via the FFT:

In [ ]:
import os, importlib

examples_dir = os.path.dirname(importlib.import_module('pysndlib.examples').__file__)
example = os.path.join(examples_dir, 'flute_trill_1.wav')
print('example file:', example)
dsp.down_oct(1, example, 0, 'flute_down.wav')
dd, _ = clm.file2ndarray('flute_down.wav')
print('output shape:', dd.shape)

In [ ]:
ipd.Audio('flute_down.wav', rate=SR)

## Summary

- Filters are generators too: make once, call per sample with the input
- `formant` banks shape noise into vowels
- Delays feed effects; `dsp` has the classic designs ready-made

Next: [writing and using instruments](04_Instruments.ipynb)